In [0]:
from pyspark.sql import functions as F

TARGET = "workspace.open_meteo.silver_city_forecast_enriched"

grid_map = spark.table("workspace.open_meteo.silver_city_grid_map")
enriched_fc = spark.table("workspace.open_meteo.silver_hourly_forecast_enriched")

city_forecast = (
    enriched_fc.join(
        F.broadcast(grid_map),
        (enriched_fc.latitude == grid_map.fc_lat)
        & (enriched_fc.longitude == grid_map.fc_lon),
        how="inner",
    )
    .drop("latitude", "longitude")
)

(
    city_forecast.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)